# Penjelasan Model (Model-agnostic Explanations)

Course 3: Supervised Learning, Klasifikasi

Model yang kompleks seperti random forest sering akurat tapi sulit dijelaskan, jadi disebut **black-box**. Di notebook ini aku melatih random forest untuk memprediksi apakah seorang karyawan sedang mencari pekerjaan baru, lalu menjelaskan cara kerjanya dengan metode yang tidak bergantung pada jenis modelnya (model-agnostic):

1. **Permutation feature importance**: fitur mana yang paling penting
2. **Partial dependence plot (PDP)**: bagaimana perubahan satu fitur memengaruhi prediksi
3. **Global surrogate model**: model sederhana yang meniru black-box secara keseluruhan
4. **LIME**: model sederhana yang meniru black-box di sekitar satu data saja (penjelasan lokal)

## Dataset

Datanya `hr_new_job_processed.csv`, profil karyawan yang sudah diproses (semua fitur numerik), dengan kolom `target`: `1` kalau karyawan mencari pekerjaan baru dan `0` kalau tidak. Fiturnya antara lain indeks pembangunan kota (`city_development_index`), jam pelatihan (`training_hours`), ukuran perusahaan (`company_size`), pengalaman (`experience`), dan tingkat pendidikan (`education_level`). Dibaca langsung dari link:

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/hr_new_job_processed.csv

## 1. Import dan baca data

In [ ]:
import sys, subprocess, warnings
try:
    import lime.lime_tabular
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lime"])
    import lime.lime_tabular

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RS = 123

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/hr_new_job_processed.csv"
job = pd.read_csv(URL)
print(job.shape)
job.describe().T.round(2)

## 2. Model black-box: random forest

In [ ]:
X = job.drop(columns="target")
y = job["target"]
print((y.value_counts(normalize=True) * 100).round(1).to_string())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=12)
print("train:", X_train.shape, "| test:", X_test.shape)

In [ ]:
black_box = RandomForestClassifier(n_estimators=100, max_depth=25, max_features=10,
                                   bootstrap=True, random_state=RS)
black_box.fit(X_train, y_train)

y_bb = black_box.predict(X_test)
print("akurasi:", round(accuracy_score(y_test, y_bb), 4))
print("AUC    :", round(roc_auc_score(y_test, black_box.predict_proba(X_test)[:, 1]), 4))

Random forest ini terdiri dari 100 pohon dengan kedalaman sampai 25, jadi tidak mungkin dibaca langsung. Metode-metode berikut bisa dipakai untuk model jenis apa pun, jadi kalau modelnya diganti (misalnya XGBoost), langkah penjelasannya tetap sama.

## 3. Permutation feature importance

Idenya: acak (permutasi) nilai satu fitur di data, lalu lihat seberapa turun performa model. Kalau performa turun banyak, berarti model sangat bergantung pada fitur itu. Aku hitung di **data test** supaya yang terukur adalah kepentingan fitur untuk data baru, dan `n_repeats=10` agar hasilnya lebih stabil.

In [ ]:
perm = permutation_importance(black_box, X_test, y_test, n_repeats=10, random_state=RS, n_jobs=-1)
print("bentuk hasil (fitur x pengulangan):", perm.importances.shape)

urut = perm.importances_mean.argsort()
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(perm.importances[urut].T, vert=False)
ax.set_yticklabels(X_test.columns[urut])
ax.set_title("Permutation importance (data test)")
ax.set_xlabel("penurunan akurasi saat fitur diacak")
plt.tight_layout()
plt.show()

In [ ]:
penting = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("fitur paling penting:", ", ".join(penting.index[:5]))
penting.round(4).head(8)

Kotak yang lebih ke kanan berarti fitur lebih penting. Jika sebuah fitur penting, mengacaknya membuat akurasi turun jelas. Fitur dengan nilai mendekati nol tidak banyak dipakai model.

## 4. Partial dependence plot (PDP)

PDP menunjukkan efek rata-rata sebuah fitur terhadap peluang prediksi, dengan nilai fitur lain dirata-ratakan. PDP biasanya dibuat untuk fitur-fitur terpenting saja, jadi aku ambil dua fitur teratas dari permutation importance.

In [ ]:
top2 = penting.index[:2].tolist()
fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(black_box, X_train, features=top2, random_state=RS, ax=ax)
plt.show()

Kurva yang menurun berarti makin besar nilai fitur, makin rendah peluang karyawan mencari pekerjaan baru, dan sebaliknya untuk kurva yang naik. Perhatikan juga seberapa curam dan linear kurvanya.

### Latihan: PDP untuk fitur penting lainnya

In [ ]:
lainnya = penting.index[2:5].tolist()
fig, ax = plt.subplots(figsize=(14, 4))
PartialDependenceDisplay.from_estimator(black_box, X_train, features=lainnya, random_state=RS, ax=ax)
plt.show()

## 5. Global surrogate model

Surrogate adalah model sederhana yang dilatih untuk **meniru keluaran black-box**. Langkahnya:

1. Ambil sekumpulan data `X_test` sebagai masukan
2. Minta black-box memprediksinya (`y_bb`)
3. Latih model sederhana dengan `X_test` sebagai masukan dan `y_bb` sebagai label
4. Ukur seberapa mirip surrogate dengan black-box (**fidelity**), lalu baca surrogate-nya

Yang ditiru adalah prediksi black-box, bukan label sebenarnya.

### Surrogate: regresi logistik (L1)

In [ ]:
scaler = StandardScaler().fit(X_test)
X_test_s = scaler.transform(X_test)

lr_surr = LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000, random_state=RS)
lr_surr.fit(X_test_s, y_bb)

fidelity_lr = accuracy_score(y_bb, lr_surr.predict(X_test_s))
print(f"fidelity (kemiripan dengan black-box): {fidelity_lr:.3f}")

In [ ]:
koef = pd.Series(lr_surr.coef_[0], index=X_test.columns)
koef = koef[koef.abs() >= 0.01].sort_values()
koef.plot.barh(color=["tab:red" if v < 0 else "tab:green" for v in koef], figsize=(7, 4))
plt.title("Koefisien surrogate regresi logistik")
plt.xlabel("koefisien (fitur terstandardisasi)")
plt.show()
print("menaikkan peluang :", ", ".join(koef.index[::-1][:3]))
print("menurunkan peluang:", ", ".join(koef.index[:3]))

Koefisien positif (hijau) mendorong prediksi ke arah "mencari pekerjaan baru", yang negatif (merah) sebaliknya. Karena fiturnya distandardisasi, besar koefisien bisa dibandingkan antar fitur. Tapi ingat, ini penjelasan dari model tiruan. Seberapa bisa dipercaya bergantung pada nilai fidelity di atas.

### Latihan: surrogate decision tree

In [ ]:
tree_surr = DecisionTreeClassifier(max_depth=5, max_features=10, random_state=RS)
tree_surr.fit(X_test, y_bb)

fidelity_tree = accuracy_score(y_bb, tree_surr.predict(X_test))
print(f"fidelity decision tree : {fidelity_tree:.3f}")
print(f"fidelity regresi logistik: {fidelity_lr:.3f}")

In [ ]:
print(export_text(tree_surr, feature_names=list(X_test.columns), max_depth=3))

Aturan di atas adalah penjelasan global yang bisa dibaca manusia. Kalau fidelity-nya rendah, berarti surrogate terlalu sederhana untuk meniru black-box dan penjelasannya kurang bisa diandalkan.

## 6. LIME: penjelasan lokal

Surrogate global bisa kurang tepat kalau datanya terdiri dari beberapa kelompok yang berbeda. **LIME** menjelaskan satu prediksi saja: ia membuat banyak data tiruan di sekitar data itu, meminta black-box memprediksinya, lalu melatih model sederhana yang berbobot (data yang lebih dekat dianggap lebih penting). Koefisien model sederhana itulah penjelasannya.

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    mode="classification",
    class_names=["Tidak pindah", "Pindah"],
    feature_names=list(X_train.columns),
    random_state=RS,
)

def prediksi_proba(data):
    # LIME mengirim array numpy, jadi dibungkus lagi menjadi DataFrame berkolom
    return black_box.predict_proba(pd.DataFrame(data, columns=X_train.columns))

In [ ]:
def jelaskan(posisi):
    baris = X_test.iloc[posisi]
    p = black_box.predict_proba(X_test.iloc[[posisi]])[0]
    print(f"data ke-{posisi}: peluang tidak pindah = {p[0]:.2f}, peluang pindah = {p[1]:.2f}")
    print(baris.round(2).to_string())
    exp = explainer.explain_instance(baris.values, prediksi_proba, num_features=10)
    exp.as_pyplot_figure()
    plt.tight_layout()
    plt.show()
    return exp

# pilih satu karyawan yang diprediksi tidak pindah
posisi_a = int(np.where(y_bb == 0)[0][0])
exp_a = jelaskan(posisi_a)

Grafik LIME menampilkan fitur dan rentang nilainya yang paling berpengaruh pada prediksi data ini. Batang ke satu arah mendorong prediksi ke "pindah", dan batang ke arah sebaliknya mendorong ke "tidak pindah".

### Latihan: jelaskan data lain

In [ ]:
# kali ini karyawan yang diprediksi pindah
posisi_b = int(np.where(y_bb == 1)[0][0])
exp_b = jelaskan(posisi_b)

In [ ]:
print("alasan utama prediksi data pertama (tidak pindah):")
for aturan, bobot in exp_a.as_list()[:3]:
    print(f"  {aturan:<40} {bobot:+.3f}")
print("alasan utama prediksi data kedua (pindah):")
for aturan, bobot in exp_b.as_list()[:3]:
    print(f"  {aturan:<40} {bobot:+.3f}")

## Ringkasan

| Metode | Cakupan | Menjawab pertanyaan |
| --- | --- | --- |
| Permutation importance | global | fitur mana yang paling berpengaruh pada performa model |
| PDP | global | bagaimana perubahan satu fitur memengaruhi prediksi |
| Global surrogate | global | bisakah model sederhana meniru black-box, dan apa aturannya |
| LIME | lokal | kenapa model memprediksi seperti itu untuk satu data |

Semua metode ini model-agnostic, tapi masing-masing punya keterbatasan: permutation importance bisa menyesatkan untuk fitur yang saling berkorelasi, surrogate hanyalah tiruan, dan LIME bisa berbeda hasilnya di tiap pengacakan.